1. Install and Import

In [1]:
!pip install pandas scikit-learn sentence-transformers

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached torch-2.13.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached cuda_toolkit-13.0.3.0-py2.py3-none-any.whl.metadata (17 kB)
  Using cached cuda_bindings-13.3.1-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.5 kB)
  Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_2

In [2]:
import pandas as pd
import numpy as np
import re
from itertools import combinations
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. Load files

In [3]:
# File names

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

INSTANCE_PATH = "instance_orgname.txt"
CLUSTER_PATH = "clusters.txt"

# Load InstanceID and organization name
instances = pd.read_csv(
    INSTANCE_PATH,
    sep="\t",
    header=None,
    names=["InstanceID", "OrgName"],
    dtype={"InstanceID": str, "OrgName": str},
    encoding="utf-8"
)

# Load GroupID and clustered InstanceIDs
clusters = pd.read_csv(
    CLUSTER_PATH,
    sep="\t",
    header=None,
    names=["GroupID", "InstanceIDs"],
    dtype={"GroupID": str, "InstanceIDs": str},
    encoding="utf-8"
)

# Convert clusters from wide format to long format
cluster_long = (
    clusters
    .assign(InstanceID=clusters["InstanceIDs"].str.split("|"))
    .explode("InstanceID")
    .reset_index(drop=True)
)

cluster_long = cluster_long[["GroupID", "InstanceID"]]

# Merge organization names with GroupID
data = instances.merge(cluster_long, on="InstanceID", how="left")

# Check basic counts
print("Number of instances:", len(data))
print("Number of clusters:", data["GroupID"].nunique())
print("Missing GroupID:", data["GroupID"].isna().sum())

data.head()

Number of instances: 1424
Number of clusters: 990
Missing GroupID: 0


,InstanceID,OrgName,GroupID
0,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001
1,100000002,3IA Côte d’Azur,10000002
2,100000003,3IA Côte d'Azur,10000002
3,100000004,5511 Science and Technology Innovation Talent ...,10000003
4,100000005,FP7 European Commission 7th Framework,10000004


3. Generate all positive pairs

In [4]:
def make_pair(row1, row2, label, negative_type=None):
    # Create a pairwise record from two rows
    return {
        "InstanceID_1": row1["InstanceID"],
        "OrgName_1": row1["OrgName"],
        "GroupID_1": row1["GroupID"],
        "InstanceID_2": row2["InstanceID"],
        "OrgName_2": row2["OrgName"],
        "GroupID_2": row2["GroupID"],
        "Label": label,
        "NegativeType": negative_type
    }


positive_pairs = []

# Generate all within-cluster combinations
for group_id, group_df in data.groupby("GroupID"):
    if len(group_df) < 2:
        continue
    
    records = group_df.to_dict("records")
    
    for row1, row2 in combinations(records, 2):
        positive_pairs.append(make_pair(row1, row2, label=1, negative_type=None))

positive_df = pd.DataFrame(positive_pairs)

print("Number of positive pairs:", len(positive_df))
positive_df.head()

Number of positive pairs: 566


,InstanceID_1,OrgName_1,GroupID_1,InstanceID_2,OrgName_2,GroupID_2,Label,NegativeType
0,100000002,3IA Côte d’Azur,10000002,100000003,3IA Côte d'Azur,10000002,1,None
1,100000005,FP7 European Commission 7th Framework,10000004,100000006,7th Framework Programme,10000004,1,None
2,100000007,Action for AIDS,10000005,100000008,ACTION FOR AIDS (AfA SINGAPORE),10000005,1,None
3,100000009,Austrian Academy of sciences APART-MINT,10000006,100000010,APART-MINT,10000006,1,None
4,100000011,Centre of Excellence for Environmental Decisio...,10000007,100000012,ARC Centre of Excellence for Environmental Dec...,10000007,1,None


4. Helper functions for negative sampling

In [5]:
def pair_key(id1, id2):
    # Create an order-independent key for deduplication
    return tuple(sorted([str(id1), str(id2)]))


def is_negative(row1, row2):
    # Return True if two rows belong to different clusters
    return row1["GroupID"] != row2["GroupID"]


def add_negative_pair(candidate_pairs, row1, row2, negative_type, used_pairs):
    # Add a negative pair if it is valid and not already used
    key = pair_key(row1["InstanceID"], row2["InstanceID"])
    
    if key in used_pairs:
        return
    
    if row1["GroupID"] == row2["GroupID"]:
        return
    
    candidate_pairs.append(make_pair(row1, row2, label=0, negative_type=negative_type))
    used_pairs.add(key)

5. Random negative pairs

In [6]:
N_POSITIVE = len(positive_df)
N_RANDOM_NEG = N_POSITIVE

used_pairs = set(
    pair_key(row["InstanceID_1"], row["InstanceID_2"])
    for _, row in positive_df.iterrows()
)

random_negatives = []

records = data.to_dict("records")

while len(random_negatives) < N_RANDOM_NEG:
    # Randomly sample two different records
    idx1, idx2 = np.random.choice(len(records), size=2, replace=False)
    row1 = records[idx1]
    row2 = records[idx2]
    
    add_negative_pair(
        candidate_pairs=random_negatives,
        row1=row1,
        row2=row2,
        negative_type="random",
        used_pairs=used_pairs
    )

random_negative_df = pd.DataFrame(random_negatives)

print("Random negatives:", len(random_negative_df))
random_negative_df.head()

Random negatives: 566


,InstanceID_1,OrgName_1,GroupID_1,InstanceID_2,OrgName_2,GroupID_2,Label,NegativeType
0,100001186,Tatung University,10000826,100000678,Korea National Institute of Health,10000472,0,random
1,100000027,Agencia Nacional de Investigación y Desarrollo...,10000018,100000327,EGI,10000228,0,random
2,100001277,Universitat de Barcelona,10000885,100000458,HES-SO University of Applied Sciences and Arts...,10000321,0,random
3,100001270,Universidade Nova de Lisboa,10000881,100000483,Home Office,10000341,0,random
4,100001058,Sansom Institute of Health Research,10000729,100001404,Yunnan Science and Technology Fund,10000976,0,random


6. Token-blocking negative pairs

In [7]:
'''token blocking: 
1. last token
2. first token
3. important token after removing generic words
4. acronym'''

N_TOKEN_NEG = N_POSITIVE * 2

GENERIC_TOKENS = {
    "the", "of", "for", "and", "in", "on", "at", "by", "to",
    "university", "institute", "center", "centre", "foundation",
    "fund", "council", "agency", "department", "ministry",
    "research", "science", "sciences", "technology", "national",
    "program", "project", "office", "college", "school"
}


def normalize_text(text):
    # Convert text to lowercase
    text = str(text).lower()
    
    # Replace non-alphanumeric characters with spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def tokenize(text):
    # Tokenize normalized text
    return normalize_text(text).split()


def get_acronym(text):
    # Create a simple acronym from tokens
    tokens = tokenize(text)
    if not tokens:
        return ""
    return "".join(token[0] for token in tokens if token)


def get_important_token(text):
    # Select the longest non-generic token as an important token
    tokens = [t for t in tokenize(text) if t not in GENERIC_TOKENS]
    if not tokens:
        return ""
    return sorted(tokens, key=len, reverse=True)[0]


def get_token_block_keys(text):
    # Generate multiple token-based blocking keys
    tokens = tokenize(text)
    keys = set()
    
    if tokens:
        keys.add("first:" + tokens[0])
        keys.add("last:" + tokens[-1])
    
    important = get_important_token(text)
    if important:
        keys.add("important:" + important)
    
    acronym = get_acronym(text)
    if len(acronym) >= 2:
        keys.add("acronym:" + acronym)
    
    return keys


# Build block key index
block_to_indices = {}

for idx, row in data.iterrows():
    keys = get_token_block_keys(row["OrgName"])
    
    for key in keys:
        block_to_indices.setdefault(key, []).append(idx)


# Generate token-blocking negative candidates
token_negatives = []

for block_key, indices in block_to_indices.items():
    if len(token_negatives) >= N_TOKEN_NEG:
        break
    
    if len(indices) < 2:
        continue
    
    # Limit very large blocks to avoid excessive pair generation
    # This keeps the sampling practical and reproducible
    if len(indices) > 200:
        indices = list(np.random.choice(indices, size=200, replace=False))
    
    for idx1, idx2 in combinations(indices, 2):
        if len(token_negatives) >= N_TOKEN_NEG:
            break
        
        row1 = data.loc[idx1].to_dict()
        row2 = data.loc[idx2].to_dict()
        
        add_negative_pair(
            candidate_pairs=token_negatives,
            row1=row1,
            row2=row2,
            negative_type="token_blocking",
            used_pairs=used_pairs
        )

token_negative_df = pd.DataFrame(token_negatives)

print("Token-blocking negatives:", len(token_negative_df))
token_negative_df.head()

Token-blocking negatives: 1132


,InstanceID_1,OrgName_1,GroupID_1,InstanceID_2,OrgName_2,GroupID_2,Label,NegativeType
0,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100000109,Beijing Higher Education Young Elite Teacher P...,10000074,0,token_blocking
1,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100000205,China Hubei Province Soft Science Key Project,10000144,0,token_blocking
2,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100000211,China Postdoctoral Science Foundation-funded p...,10000148,0,token_blocking
3,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100000216,China Yunnan province major science and techno...,10000151,0,token_blocking
4,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100000219,Chongqing Language Research Project,10000154,0,token_blocking


7. Embedding nearest-neighbor negative pairs

In [8]:
N_EMBED_NEG = N_POSITIVE * 2

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Load multilingual embedding model
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Encode organization names
embeddings = model.encode(
    data["OrgName"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Find nearest neighbors using cosine distance
# Since embeddings are normalized, cosine distance is appropriate
n_neighbors = 20

nn = NearestNeighbors(
    n_neighbors=n_neighbors,
    metric="cosine"
)

nn.fit(embeddings)

distances, neighbor_indices = nn.kneighbors(embeddings)

embedding_negatives = []

for i in range(len(data)):
    if len(embedding_negatives) >= N_EMBED_NEG:
        break
    
    row1 = data.iloc[i].to_dict()
    
    for neighbor_rank in range(1, n_neighbors):
        if len(embedding_negatives) >= N_EMBED_NEG:
            break
        
        j = neighbor_indices[i][neighbor_rank]
        row2 = data.iloc[j].to_dict()
        
        add_negative_pair(
            candidate_pairs=embedding_negatives,
            row1=row1,
            row2=row2,
            negative_type="embedding_nn",
            used_pairs=used_pairs
        )

embedding_negative_df = pd.DataFrame(embedding_negatives)

print("Embedding nearest-neighbor negatives:", len(embedding_negative_df))
embedding_negative_df.head()

Batches: 100%|██████████| 23/23 [00:02<00:00,  8.75it/s]


Embedding nearest-neighbor negatives: 1132


,InstanceID_1,OrgName_1,GroupID_1,InstanceID_2,OrgName_2,GroupID_2,Label,NegativeType
0,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100001144,Special Innovation Project of University in Gu...,10000795,0,embedding_nn
1,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100000068,Applied Basic and Advanced Technology Research...,10000044,0,embedding_nn
2,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100000544,Innovation Team Project of Colleges in Guangdo...,10000375,0,embedding_nn
3,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100001222,Tianjin Science and Technology Committee,10000849,0,embedding_nn
4,100000001,2021 Tianjin Postgraduate Research and Innovat...,10000001,100001223,Tianjin Municipal Science and Technology Commi...,10000849,0,embedding_nn


8. Combine final pairwise gold dataset

In [9]:
pairwise_gold_df = pd.concat(
    [
        positive_df,
        random_negative_df,
        token_negative_df,
        embedding_negative_df
    ],
    ignore_index=True
)

# Shuffle final dataset
pairwise_gold_df = pairwise_gold_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("Final dataset size:", len(pairwise_gold_df))
print(pairwise_gold_df["Label"].value_counts())
print(pairwise_gold_df["NegativeType"].value_counts(dropna=False))

pairwise_gold_df.head()

Final dataset size: 3396
Label
0    2830
1     566
Name: count, dtype: int64
NegativeType
embedding_nn      1132
token_blocking    1132
None               566
random             566
Name: count, dtype: int64


,InstanceID_1,OrgName_1,GroupID_1,InstanceID_2,OrgName_2,GroupID_2,Label,NegativeType
0,100000691,Lancaster University,10000483,100000692,Universidade de Lancaster,10000483,1,None
1,100000046,Alfred Sloan Fellowship,10000029,100000236,"CoMotion, University of Washington",10000165,0,embedding_nn
2,100000659,Key Science and Technology Program of Shaanxi ...,10000458,100001133,Social science funds of Jiangsu province,10000785,0,token_blocking
3,100000994,"PowerBI Co., Ltd Belgium",10000683,100000995,Power BI,10000683,1,None
4,100001156,Universitas Negeri Semarang,10000804,100001157,State University of Semarang,10000804,1,None


9. Save

In [10]:
pairwise_gold_df.to_csv(
    "pairwise_gold_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

positive_df.to_csv(
    "positive_pairs_all.csv",
    index=False,
    encoding="utf-8-sig"
)

random_negative_df.to_csv(
    "negative_random.csv",
    index=False,
    encoding="utf-8-sig"
)

token_negative_df.to_csv(
    "negative_token_blocking.csv",
    index=False,
    encoding="utf-8-sig"
)

embedding_negative_df.to_csv(
    "negative_embedding_nn.csv",
    index=False,
    encoding="utf-8-sig"
)

10. Dataset check

In [11]:
# Check duplicate pairs
pairwise_gold_df["PairKey"] = pairwise_gold_df.apply(
    lambda row: pair_key(row["InstanceID_1"], row["InstanceID_2"]),
    axis=1
)

print("Duplicate pair count:", pairwise_gold_df["PairKey"].duplicated().sum())

# Check label distribution
print(pairwise_gold_df["Label"].value_counts())

# Check negative type distribution
print(pairwise_gold_df["NegativeType"].value_counts(dropna=False))

Duplicate pair count: 0
Label
0    2830
1     566
Name: count, dtype: int64
NegativeType
embedding_nn      1132
token_blocking    1132
None               566
random             566
Name: count, dtype: int64


11. Create 10 stratified bins

In [12]:
from sklearn.model_selection import StratifiedKFold

# Load pairwise gold dataset
df = pd.read_csv("pairwise_gold_dataset.csv", encoding="utf-8-sig")

print(df["Label"].value_counts())
print(df["NegativeType"].value_counts(dropna=False))

# Create 10 stratified bins based on Label
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_SEED
)

df["Bin"] = -1

for bin_id, (_, test_index) in enumerate(skf.split(df, df["Label"])):
    df.loc[test_index, "Bin"] = bin_id

print(pd.crosstab(df["Bin"], df["Label"]))

# Save the dataset with Bin assignments for ML, CNN, and BERT
df.to_csv(
    "pairwise_gold_dataset_with_bins.csv",
    index=False,
    encoding="utf-8-sig"
)

Label
0    2830
1     566
Name: count, dtype: int64
NegativeType
embedding_nn      1132
token_blocking    1132
NaN                566
random             566
Name: count, dtype: int64
Label    0   1
Bin           
0      283  57
1      283  57
2      283  57
3      283  57
4      283  57
5      283  57
6      283  56
7      283  56
8      283  56
9      283  56
